# 004 · Debugging with pdb

> **Standard Library › Testing & Debugging** · Python 3.14

`pdb` is Python's built-in, command-line debugger: breakpoints, stepping, stack inspection and evaluating expressions in any frame. Most of the time you will drive the same machinery through a graphical debugger (VS Code, PyCharm, JupyterLab), but knowing `pdb` lets you debug anywhere — over SSH, in a container, or in a CI log — and its post-mortem mode is a superpower.

**In this notebook you will learn:**

- how `breakpoint()` relates to `Debugger.Break()` and IDE breakpoints
- the essential `pdb` commands and their Visual Studio equivalents (F10, F11, F5, …)
- post-mortem debugging: `python -m pdb`, `pdb.pm()`, and Jupyter's `%debug`
- how to disable or redirect `breakpoint()` with `PYTHONBREAKPOINT` and `sys.breakpointhook`
- non-interactive tools that work everywhere: `traceback`, `inspect`, `faulthandler`, and logging vs print debugging

> ⚠️ This notebook never starts the interactive debugger in an executed cell — a `(Pdb)` prompt waiting for input would hang the notebook. Debugger sessions are shown as text instead.

## .NET mental model

| .NET / Visual Studio | Python |
|---|---|
| IDE breakpoint (F9) | IDE breakpoint (VS Code, PyCharm, JupyterLab — all built on the same tracing hooks) |
| `System.Diagnostics.Debugger.Break()` | `breakpoint()` (calls `sys.breakpointhook()` → `pdb.set_trace()` by default) |
| `Debugger.IsAttached` | no reliable equivalent — since 3.14 `breakpoint()` and `python -m pdb` use the low-overhead `sys.monitoring` API, so `sys.gettrace()` no longer reveals them |
| Visual Studio debugger UI | `pdb` text UI, or a graphical front end (`debugpy` in VS Code) |
| Attach to process (Ctrl+Alt+P) | `python -m pdb -p PID` (new in 3.14), or VS Code "attach" via `debugpy` |
| Crash dump analysis (`dotnet-dump analyze`) | post-mortem: `python -m pdb script.py`, `pdb.pm()`, `%debug` |
| `Exception.StackTrace`, `new StackTrace()` | `traceback.format_exc()`, `traceback.extract_stack()` |
| `System.Reflection`, `[CallerMemberName]` | `inspect` (`stack()`, `currentframe()`, `signature()`, `getsource()`) |

## `breakpoint()` — the `Debugger.Break()` of Python

Put `breakpoint()` on any line; when execution reaches it, you get a `(Pdb)` prompt in the terminal *in that frame*. It is the built-in (since 3.7) replacement for the older `import pdb; pdb.set_trace()` idiom.

```csharp
// C#
if (order.Total < 0)
    System.Diagnostics.Debugger.Break();   // stops only if a debugger is attached
```

```python
# Python — shop.py (run in a terminal, not here)
def order_total(lines):
    total = 0
    for sku, price, qty in lines:
        breakpoint()                  # pauses here and opens the (Pdb) prompt
        total += line_total(price, qty)
    return total
```

⚠️ **Gotcha:** unlike `Debugger.Break()`, which is ignored when no debugger is attached, `breakpoint()` **always** stops and waits for input. A forgotten `breakpoint()` in production code will hang a server or a CI job — set `PYTHONBREAKPOINT=0` there (see below) and lint for it.

## The essential pdb commands

Every command has a long name and a short abbreviation. Pressing Enter on an empty line repeats the last command (handy for repeated `n`).

| pdb command | What it does | Visual Studio equivalent |
|---|---|---|
| `n` (next) | run the current line, stepping **over** calls | F10 Step Over |
| `s` (step) | step **into** the call on the current line | F11 Step Into |
| `r` (return) | run until the current function returns | Shift+F11 Step Out |
| `c` (continue) | run until the next breakpoint | F5 Continue |
| `unt N` (until) | run until line N (or past the current loop) | Ctrl+F10 Run to Cursor |
| `j N` (jump) | set the next line to execute | Ctrl+Shift+F10 Set Next Statement |
| `l` / `ll` (list / longlist) | show source around the line / the whole function | the source editor |
| `p expr` / `pp expr` | print / pretty-print an expression | QuickWatch, Immediate window |
| `a` (args) | print the current function's arguments | Locals window |
| `display expr` | re-print `expr` whenever it changes | Watch window |
| `w` (where) | print the call stack | Call Stack window |
| `u` / `d` (up / down) | move to the caller / callee frame | double-click a Call Stack frame |
| `b file:line` / `b func` | set a breakpoint (`b 12, qty > 1` for a condition) | F9 / conditional breakpoint |
| `tbreak ...` | one-shot breakpoint | "Run to Cursor"-like temporary breakpoint |
| `cl N` (clear) | delete breakpoint number N (`cl` alone = all) | Ctrl+Shift+F9 |
| `b` (no args) | list breakpoints | Breakpoints window |
| `interact` | open a full Python REPL with the frame's variables | Immediate window |
| `!stmt` | execute a statement, e.g. `!qty = 3` | edit a value in Locals |
| `q` (quit) | abort the program | Shift+F5 Stop Debugging |
| `h` / `h cmd` (help) | list commands / help for one command | — |

**Tip:** if a variable name clashes with a command (a variable called `n`, `c`, `p`…), print it with `p n` or `!n` — typing `n` alone runs the *next* command.

## What a session looks like

Here is a real, lightly trimmed session against the `shop.py` above. The program contains a bug: one price is the string `"0.75"`. Lines starting with `(Pdb)` show what was typed.

```text
$ python shop.py
> shop.py(8)order_total()
-> breakpoint()
(Pdb) p sku, price, qty
('apple', 0.5, 4)
(Pdb) n
> shop.py(9)order_total()
-> total += line_total(price, qty)
(Pdb) s
--Call--
> shop.py(1)line_total()
-> def line_total(price, qty):
(Pdb) ll
  1  ->	def line_total(price, qty):
  2  	    return price * qty
(Pdb) u
> shop.py(9)order_total()
-> total += line_total(price, qty)
(Pdb) p total
0
(Pdb) d
> shop.py(1)line_total()
-> def line_total(price, qty):
(Pdb) r
--Return--
> shop.py(2)line_total()->2.0
-> return price * qty
(Pdb) c
> shop.py(8)order_total()
-> breakpoint()
(Pdb) pp locals()
{'lines': [('apple', 0.5, 4), ('pear', '0.75', 2)],
 'price': '0.75',
 'qty': 2,
 'sku': 'pear',
 'total': 2.0}
```

The `>` line is "file(line)function()", and `->` shows the line that is **about to** execute.

## Post-mortem debugging

Often you don't know where to put a breakpoint — the program just crashes. **Post-mortem** debugging opens the debugger *in the frame where the exception was raised*, after the fact, with all local variables intact. It is the Python equivalent of opening a crash dump, except it is instant and fully interactive.

```text
$ python -m pdb -c continue shop.py      # run under pdb; stop only if it crashes
Traceback (most recent call last):
  ...
  File "shop.py", line 9, in order_total
    total += line_total(price, qty)
TypeError: unsupported operand type(s) for +=: 'float' and 'str'
Uncaught exception. Entering post mortem debugging
Running 'cont' or 'step' will restart the program
> shop.py(9)order_total()
-> total += line_total(price, qty)
(Pdb) p price, qty
('0.75', 2)
```

Other ways in:

| Where | How |
|---|---|
| Interactive `python` REPL, after a traceback | `import pdb; pdb.pm()` (uses `sys.last_exc`) |
| Inside your own `except` block | `pdb.post_mortem()` (uses the exception being handled) |
| Jupyter / IPython, after a cell raised | run `%debug` in the next cell |
| Jupyter, automatically on every exception | `%pdb on` |
| From the start, under the debugger | `python -m pdb script.py` (or `-m package.module`) |
| An already-running process (3.14+) | `python -m pdb -p PID` |

The `-c continue` part matters: without it, `python -m pdb` stops before the first line of the script, like starting with F11 instead of F5.

## Debugging in Jupyter and VS Code

- **`%debug`** — after a cell raises, run `%debug` in a new cell to get an `ipdb>` prompt in the frame of the exception (IPython's enhanced pdb: colours, tab completion). Type `q` to leave.
- **`%pdb on`** — enter the post-mortem debugger automatically whenever a cell raises.
- **Visual debuggers** — VS Code (Python + Jupyter extensions) has *Debug Cell* and full breakpoint/step/watch support in notebooks and `.py` files, via the `debugpy` adapter. JupyterLab has a built-in debugger panel (the bug icon) when the kernel supports it (`ipykernel` 6+). These feel like Visual Studio: click in the gutter, F10/F11/F5.

```text
In [7]: order_total([("pear", "0.75", 2)])
TypeError: unsupported operand type(s) for +=: 'int' and 'str'

In [8]: %debug
> /tmp/ipykernel_.../....py(4)order_total()
ipdb> p price
'0.75'
ipdb> q
```

**Tip:** for day-to-day work, use the graphical debugger. Use `pdb` when you are on a remote machine, inside a container, or the bug only reproduces from the command line.

## Disabling and redirecting `breakpoint()`

`breakpoint()` does not call `pdb` directly; it calls `sys.breakpointhook()`. The default hook consults the **`PYTHONBREAKPOINT`** environment variable on every call:

| `PYTHONBREAKPOINT` | Effect |
|---|---|
| unset or empty | `pdb.set_trace()` |
| `0` | `breakpoint()` does nothing — safe for production/CI |
| `some.module.callable` | imports and calls that instead (e.g. a third-party debugger such as `ipdb.set_trace`) |

```bash
PYTHONBREAKPOINT=0 python shop.py        # all breakpoint() calls become no-ops
```

We can prove the `0` setting safely by running a child Python process that calls `breakpoint()` (with input from `/dev/null` and a timeout, so it could never hang this notebook even if the setting failed):

In [ ]:
import os
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-c", "breakpoint(); print('ran straight past breakpoint()')"],
    env={**os.environ, "PYTHONBREAKPOINT": "0"},
    stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=30,
)
print(result.returncode, result.stdout.strip())

Inside a process you can also replace the hook itself. Here a harmless hook prints the caller's local variables instead of stopping — a tiny "log-point", like Visual Studio's *Actions* breakpoints that print a message and continue:

In [ ]:
import inspect

def logpoint(*args, **kwargs):
    caller = inspect.currentframe().f_back           # the frame that called breakpoint()
    print(f"breakpoint in {caller.f_code.co_name}() line {caller.f_lineno}: {dict(caller.f_locals)}")

def line_total(price, qty):
    return price * qty

def order_total(lines):
    total = 0
    for sku, price, qty in lines:
        breakpoint()                                 # calls sys.breakpointhook()
        total += line_total(price, qty)
    return total

sys.breakpointhook = logpoint
try:
    print(order_total([("apple", 0.5, 4), ("pear", 0.75, 2)]))
finally:
    sys.breakpointhook = sys.__breakpointhook__      # always restore the original
print(sys.breakpointhook is sys.__breakpointhook__)  # True

## Non-interactive inspection: `traceback`

When you can't attach a debugger (a server, a scheduled job), you capture and log the information instead. The `traceback` module formats exceptions and stacks — the equivalent of `exception.ToString()` / `exception.StackTrace` and `new StackTrace()`:

| Function | Returns / does |
|---|---|
| `traceback.print_exc()` | prints the current exception's traceback (to `stderr` by default; pass `file=`) |
| `traceback.format_exc()` | the same, as a string — ideal for logging or storing |
| `traceback.format_exception(e)` | list of lines for any exception object |
| `traceback.extract_tb(e.__traceback__)` | structured frames (`filename`, `lineno`, `name`, `line`) |
| `traceback.print_stack()` / `extract_stack()` | the *current* call stack, no exception needed |

In [ ]:
import traceback

def parse_order(text: str) -> tuple[str, int]:
    sku, qty = text.split(",")
    return sku, int(qty)

def load_orders(rows: list[str]) -> list[tuple[str, int]]:
    return [parse_order(r) for r in rows]

try:
    load_orders(["apple,2", "pear,two"])
except ValueError as e:
    report = traceback.format_exc()                      # full text, e.g. for a log
    print(report.splitlines()[0])                        # Traceback (most recent call last):
    print(report.splitlines()[-1])                       # the exception line
    for frame in traceback.extract_tb(e.__traceback__):  # structured: no file paths needed
        print(f"  in {frame.name}() line {frame.lineno}: {frame.line}")

In a script, `traceback.print_exc()` inside an `except` block prints exactly the familiar crash output but lets the program continue. In real applications prefer `logger.exception("...")`, which logs the message **and** the traceback in one call *(see Standard Library › Logging & Config › 001 · Logging Basics)*.

## Non-interactive inspection: `inspect`

`inspect` is Python's reflection-for-debugging toolkit:

- `inspect.currentframe()` / `inspect.stack()` — the live call stack (like `new StackTrace()` / `[CallerMemberName]`)
- `inspect.signature(func)` — parameters, defaults and annotations (like `MethodInfo.GetParameters()`)
- `inspect.getsource(obj)` — the source code of a function or class (no .NET equivalent without a decompiler)

In [ ]:
def audit(message: str) -> None:
    caller = inspect.stack()[1]                  # [0] is audit itself, [1] is who called it
    print(f"[{caller.function}] {message}")

def checkout(cart_id: int, *, express: bool = False) -> None:
    audit(f"checking out cart {cart_id}")

checkout(42)
print(inspect.signature(checkout))
print(list(inspect.signature(checkout).parameters))

In [ ]:
print(inspect.getsource(parse_order))      # works for notebook cells too

⚠️ **Gotcha:** `inspect.stack()` reads source files for every frame and is **slow** — fine for debugging, not for a hot path. `inspect.currentframe().f_back` is cheap. Also, frame objects keep all their local variables alive; don't store them, and `del` a frame reference when you are done with it.

## `faulthandler`: when Python itself crashes or hangs

For hard crashes (a segfault in a C extension) or deadlocks, there is no Python exception to catch. `faulthandler` dumps the Python stack of **all threads** to `stderr` when that happens — roughly what `dotnet-dump` / `dotnet-stack` give you for a stuck .NET process:

```bash
python -X faulthandler app.py            # or set PYTHONFAULTHANDLER=1
```

```python
import faulthandler
faulthandler.enable()                          # dump tracebacks on a fatal signal
faulthandler.dump_traceback_later(60)          # dump all threads if still running after 60 s
```

In [ ]:
import faulthandler
print(faulthandler.is_enabled())    # False unless -X faulthandler / PYTHONFAULTHANDLER was used

**Tip:** `python -X dev` turns on development mode — `faulthandler`, extra runtime checks, and warnings that are hidden by default. A good first step when something behaves strangely.

## Print debugging vs logging

Everyone print-debugs. Python makes it nicer with the **self-documenting f-string** `f"{expr=}"`, which prints the expression text and its value:

In [ ]:
price, qty = "0.75", 2
print(f"{price=}, {qty=}, {type(price).__name__=}")
print(f"{price * qty=}")             # aha: string repetition, not multiplication

For anything that stays in the code, use `logging` instead of `print`: messages carry a level, a timestamp and the logger name, can be switched on per module, and go to files or monitoring instead of stdout — like `ILogger<T>` in .NET.

In [ ]:
import logging

log = logging.getLogger("shop.debug-demo")
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s %(name)s: %(message)s"))
log.addHandler(handler)
log.propagate = False

log.setLevel(logging.DEBUG)
log.debug("line_total(%r, %r) -> %r", price, qty, line_total(price, qty))
log.setLevel(logging.INFO)                     # turn debug output off without editing code
log.debug("this is filtered out")
log.info("checkout complete")
log.removeHandler(handler)

**Tip:** pass values as `%s`-style arguments (`log.debug("x=%r", x)`) rather than an f-string, so the message is only formatted if the level is enabled.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Break into the debugger from code | `Debugger.Break()` | `breakpoint()` |
| Step over / into / out | F10 / F11 / Shift+F11 | `n` / `s` / `r` |
| Continue | F5 | `c` |
| Inspect a value | QuickWatch / Immediate | `p expr`, `pp expr`, `interact` |
| Call stack, change frame | Call Stack window | `w`, `u`, `d` |
| Set / clear breakpoint | F9 | `b file:line [, cond]`, `cl N` |
| Debug after a crash | crash dump analysis | `python -m pdb -c continue app.py`, `pdb.pm()`, `%debug` |
| Attach to a process | Ctrl+Alt+P | `python -m pdb -p PID` (3.14+) |
| Disable code breakpoints | — (ignored when not attached) | `PYTHONBREAKPOINT=0` |
| Exception text | `ex.ToString()` | `traceback.format_exc()` |
| Current stack | `new StackTrace()` | `inspect.stack()`, `traceback.extract_stack()` |
| Method signature | `MethodInfo.GetParameters()` | `inspect.signature(f)` |
| Stack dump on hang/crash | `dotnet-dump`, `dotnet-stack` | `faulthandler`, `python -X faulthandler` |
| Quick variable print | `Console.WriteLine($"x={x}")` | `print(f"{x=}")` |

## Exercises

1. Write a hook function `count_breakpoints` that increments a global counter instead of debugging. Install it as `sys.breakpointhook`, call a function containing `breakpoint()` in a loop five times, restore the original hook in a `finally`, and print the counter.
2. Write `def safe_divide(a, b)` that catches `ZeroDivisionError`, and returns `None` after printing the **name of the function** where the error was raised and the **last line** of the formatted traceback (use `traceback.extract_tb` and `traceback.format_exc`).
3. Write `def whoami() -> str` that returns the name of the function that called it (use `inspect.currentframe()`), and use it inside a function `process_payment()`.
4. Explain which pdb commands you would type to: (a) see where you are in the call stack, (b) look at a variable `order` in the *calling* function, (c) finish the current function and stop in the caller.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import inspect
import sys
import traceback

# 1
calls = 0

def count_breakpoints(*args, **kwargs):
    global calls
    calls += 1

def work(i):
    breakpoint()
    return i * 2

sys.breakpointhook = count_breakpoints
try:
    results = [work(i) for i in range(5)]
finally:
    sys.breakpointhook = sys.__breakpointhook__
print(calls, results)          # 5 [0, 2, 4, 6, 8]

# 2
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError as e:
        where = traceback.extract_tb(e.__traceback__)[-1].name
        print(where, "->", traceback.format_exc().splitlines()[-1])
        return None

print(safe_divide(1, 0))       # safe_divide -> ZeroDivisionError: division by zero / None

# 3
def whoami() -> str:
    return inspect.currentframe().f_back.f_code.co_name

def process_payment():
    return whoami()

print(process_payment())       # process_payment

# 4
# (a) w          — where: print the stack
# (b) u, then p order   (and d to come back down)
# (c) r          — run until the current function returns, then n to step back into the caller
```

</details>